# Part 1 — N-grams

> Part 1 of **[Tokens to Agents](../../README.md)** — the foundations of agentic AI, built from scratch.

Counting your way to a language model, and finding that the textbook fix for sparsity makes
a trigram model worse than counting single words.

The model code lives in `ngrams/`; this notebook is the narrative around it.

## Why start with counting

An n-gram model and GPT differ in exactly one place. Both answer `P(next token | context)`,
both maximise the likelihood of the next token, and both are scored by perplexity. Only the
estimator changes — counting, versus a learned function.

So this is the cheapest place to meet the ideas that get expensive later: the objective, the
metric, the context window (`n`), the Markov assumption, and the sparsity problem that
embeddings were invented to solve.

Run with `uv run jupyter lab`, or execute `run.py` for the same results as a script.

In [ ]:
from ngrams import (
    InterpolatedModel,
    NGramModel,
    close_vocabulary,
    perplexity,
    sentences,
    split,
    zero_rate,
)

from core import viz

viz.use_theme()

## The corpus

4,846 financial news sentences (Financial PhraseBank). Sentiment labels are discarded —
only the text matters here. The download is cached at the repo root and never committed.

Closing the vocabulary comes first: rare training words and unseen test words become `<unk>`.
Skip this and one unfamiliar word makes every model infinite, and you end up measuring
vocabulary coverage instead of the quality of the estimates.

In [ ]:
train, test = split(sentences())
train, test, vocab = close_vocabulary(train, test)

print(f'{len(train):,} train  {len(test):,} test  vocab {len(vocab):,}')
print(' '.join(train[0][:14]))

## The problem: most contexts have never been seen

Unsmoothed maximum likelihood assigns zero to any n-gram missing from training.
`zero_rate` reports how often that happens; perplexity reports what it costs.

In [ ]:
orders = (1, 2, 3)
mle = {n: NGramModel(n).fit(train) for n in orders}
zeros = {n: zero_rate(mle[n], test) for n in orders}

for n in orders:
    print(f'{n}-gram  zero {zeros[n]:6.1%}  perplexity {perplexity(mle[n], test):,.1f}')

In [ ]:
fig = viz.bars(
    [f'{n}-gram' for n in orders], [zeros[n] * 100 for n in orders],
    title=f'{zeros[3]:.0%} of test tokens sit in a trigram context never seen in training',
    xlabel='Test tokens given zero probability (%)', best='min', fmt='{:.1f}%',
)

The trigram is the most powerful model in principle and the one that fails hardest —
the more specific the question, the less likely it has ever been asked before.

## The textbook fix, and what it costs

Add-α smoothing pretends every possible n-gram was seen α extra times. With α = 1 —
Laplace — the infinities vanish and the numbers get much worse.

In [ ]:
alphas = (1.0, 0.1, 0.01, 0.001)
smoothed = {(n, a): NGramModel(n, alpha=a).fit(train) for n in orders for a in alphas}
ppl = {k: perplexity(m, test) for k, m in smoothed.items()}

for n in orders:
    print(f'{n}-gram  Laplace {ppl[(n, 1.0)]:9,.1f}')

The vocabulary is ~4,500 words, so `α · V` adds ~4,500 to the denominator of every
estimate. A context seen three times is competing against 4,500 units of invented
evidence. Trigram contexts are sparsest, so they lose most.

The fix is to add less:

In [ ]:
win_n, win_a = min(ppl, key=ppl.get)
fig = viz.grid_heatmap(
    [[ppl[(n, a)] for a in alphas] for n in orders],
    [f'{n}-gram' for n in orders], [f'α={a:g}' for a in alphas],
    title=(f'Laplace is the worst choice for every order — the {win_n}-gram at α={win_a:g} is '
           f'{ppl[(win_n, 1.0)] / ppl[(win_n, win_a)]:.1f}x better than its own Laplace version'),
    xlabel='Add-α smoothing strength', ylabel='Model order',
    best='min', fmt='{:,.0f}', cmap='Blues_r',
)

## Blending beats choosing

The trigram is sharp when it has seen the context and useless when it hasn't; the unigram
is blunt but always defined. Interpolation refuses the choice and lets every order vote.

In [ ]:
best_alpha = {n: min(alphas, key=lambda a: ppl[(n, a)]) for n in orders}
tuned = [smoothed[(n, best_alpha[n])] for n in orders]

weights = [(0.8, 0.15, 0.05), (0.7, 0.2, 0.1), (0.5, 0.3, 0.2),
           (0.34, 0.33, 0.33), (0.2, 0.4, 0.4), (0.1, 0.3, 0.6)]
sweep = {w: perplexity(InterpolatedModel(tuned, w), test) for w in weights}
best_w = min(sweep, key=sweep.get)

single_best = min(ppl[(n, best_alpha[n])] for n in orders)
print(f'best single    {single_best:,.1f}')
print(f'interpolated   {sweep[best_w]:,.1f}  weights {best_w}')
print(f'improvement    {1 - sweep[best_w] / single_best:.1%}')

In [ ]:
fig = viz.bars(
    [f'{n}-gram (α={best_alpha[n]:g})' for n in orders] + ['Interpolated'],
    [ppl[(n, best_alpha[n])] for n in orders] + [sweep[best_w]],
    title=f'Blending all three orders beats every single model by {1 - sweep[best_w] / single_best:.0%}',
    xlabel='Test perplexity (lower is better)', best='min', fmt='{:,.0f}',
)

## What breaks, and what fixes it

This model cannot transfer anything it learned about *"profit rose"* to *"earnings increased"*.
To counting, those share nothing — every context is an island and evidence never pools.
Smoothing decides what to say about an unseen context; it cannot help the model notice that an
unseen context *resembles* a familiar one.

That ceiling is not a tuning problem, and it is the entire motivation for Part 2: representing
words so that similar words share statistical strength.

## Takeaways

1. Unsmoothed higher-order models don't score badly — they don't score at all.
2. Laplace (α=1) is a convention, not a principled choice; on a realistic vocabulary it is
   off by two orders of magnitude and hurts most exactly where context helps most.
3. No single order wins everywhere, so blend them.

A model's behaviour on input it has never seen is set by a smoothing constant — a developer
choice, usually left at its default and rarely measured. That shape recurs all the way up
the stack, which is where this series is going.

Run `python run.py` to regenerate every figure and number in the article.